In [ ]:
import os
from pathlib import Path

# Ensure working directory is repo root regardless of how this notebook was opened
_cwd = Path(os.getcwd())
if _cwd.name in ("notebooks", "supplementary"):
    os.chdir(_cwd.parent.parent if _cwd.name == "supplementary" else _cwd.parent)


# QuantumEdge - GIC 2026 Phase 3 Reproducibility Notebook
## Dual-Timescale Quantum Reservoir Computing for Financial Volatility Forecasting

**Track:** Dynamic Systems Forecasting - Financial Volatility Prediction  
**Team:** Judith Sarjeant, Shema Nourine, Hasarindu Perera, Van Tien Nguyen

This judge-ready notebook reproduces the locked financial benchmark and adds the Phase 3 compliance analyses requested by the challenge:

- transition-event evaluation at +/-1 and +/-3 trading days;
- Diebold-Mariano tests and moving-block-bootstrap confidence intervals;
- depolarizing noise, amplitude damping, and zero-noise extrapolation;
- complete exported prediction, significance, transition, and hardware-provenance files.

The validated IBM results are loaded from the submitted evidence files. **Hardware submission is disabled by default** so a judge does not consume QPU time or require credentials. The successful primary validation was on `ibm_fez`; the independent replication was on `ibm_marrakesh`.


## Judge run order

1. Put this notebook in the submission root beside `data/`, `results/`, and `figures/`.
2. Confirm `data/market_data.csv` and `data/oxman_spx.csv` are present.
3. Leave `RUN_PROFILE = "FULL"` for the official rerun. `QUICK` is only a smoke test.
4. Leave `RUN_HARDWARE_NOW = False`. The notebook reads the submitted IBM JSON/CSV evidence.
5. Choose **Kernel -> Restart & Run All**.
6. Confirm the final audit has no failures.
7. Run Notebook 4 for the mandatory MNIST benchmark, Notebook 2 for figures, and Notebook 3 for package verification.

Generated compliance evidence:

```text
results/forecast_predictions.csv
results/forecast_significance.csv
results/transition_metrics.csv
results/transition_confusion_matrix.csv
results/amplitude_damping.csv
```


## Before running: recovery and integrity rules

- **Missing data:** confirm that the notebook is beside the complete `data/` folder.
- **Feature cache mismatch:** allow the notebook to rebuild the cache; do not rename the submitted data or cache files.
- **Memory/session constraint:** do not switch to `QUICK` for the final submission execution.
- **IBM queue delay:** do not repeatedly resubmit. Preserve each job ID from the ledger.
- **Session disconnect after submission:** retrieve the recorded job ID rather than creating a duplicate job.
- **Partial hardware completion:** the ledger identifies submitted and completed windows.
- **Existing canonical hardware files:** the notebook backs them up before replacing them.
- **Numerical drift:** report the actual Marrakesh metrics; do not alter them to match the Fez reference.
- **Unexpected failure:** preserve the full error message, executed notebook, and hardware job ledger before changing code.


## 0. Dependency check
Installs only missing packages into the current kernel. It does not upgrade packages that are already present.

In [ ]:
from pathlib import Path
import os

_START_DIR = Path.cwd().resolve()

if _START_DIR.name == "notebooks":
    PROJECT_ROOT = _START_DIR.parent
elif (_START_DIR / "notebooks").exists():
    PROJECT_ROOT = _START_DIR
else:
    # Preserve the current directory for development copies placed at root.
    PROJECT_ROOT = _START_DIR

os.chdir(PROJECT_ROOT)
print("QuantumEdge project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd().resolve())


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "scipy": "scipy",
    "pandas": "pandas",
    "qiskit": "qiskit",
    "qiskit_aer": "qiskit-aer",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "matplotlib": "matplotlib",
    "yfinance": "yfinance",
    "qiskit_ibm_runtime": "qiskit-ibm-runtime",
    "arch": "arch",
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        subprocess.run(cmd + ["--break-system-packages"], check=True)
    print("Installation complete. Restart the kernel only if the next cell reports an import error.")
else:
    print("All required core dependencies are present.")

## 1. Imports, deterministic seed, and runtime folders

In [ ]:
import os
import re
import time
import json
import math
import hashlib
import platform
import warnings
import getpass
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from scipy import stats
import statsmodels.api as sm

SEED = 7
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)

for folder in ["data", "results", "figures"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

PACKAGE_NAMES = [
    "numpy", "pandas", "qiskit", "qiskit-aer", "scikit-learn",
    "statsmodels", "matplotlib", "yfinance", "qiskit-ibm-runtime", "arch"
]
VERSIONS = {}
for package in PACKAGE_NAMES:
    try:
        VERSIONS[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        VERSIONS[package] = "not installed"

print("Python", platform.python_version(), "| seed", SEED)
print("Package versions:")
for package, version in VERSIONS.items():
    print(f"  {package}: {version}")

## 2. Configuration — use `FULL` for the submission run

In [ ]:
RUN_PROFILE = "FULL"  # QUICK = judge smoke test; change to FULL for complete reproduction
if RUN_PROFILE not in {"FULL", "QUICK"}:
    raise ValueError("RUN_PROFILE must be 'FULL' or 'QUICK'.")

FULL = RUN_PROFILE == "FULL"
CONFIG = dict(
    data_csv="data/market_data.csv",
    oxford_csv="data/oxman_spx.csv",
    allow_download=True,
    use_synthetic=False,  # synthetic data is never used silently
    window=22,
    train_frac=0.80,
    enc_low=-np.pi/2,
    enc_high=np.pi/2,
    features_cache="data/qrc_pauli_features.npy",
    features_meta="data/qrc_pauli_features.meta.json",
    force_rebuild_features=False,
    n_qubits=9,
    short=dict(J=0.3, h=1.0, p=2),
    long=dict(J=1.2, h=0.4, p=6),
    ent_keep=8,
    dt=1.0,
    feedback=True,
    gamma=0.5,
    mem_dim=3,
    headline_pauli_only=True,
    bias_correction=True,
    ridge_alphas=np.logspace(-4, 3, 25),
    study_sample=None if FULL else 900,
    scan_n=[5, 7, 9, 11] if FULL else [5, 9],
    scan_p=[2, 4] if FULL else [2],
    enc_fracs=[0.34, 0.67, 1.0] if FULL else [0.34, 1.0],
    shot_list=[256, 1024, 4096] if FULL else [256, 1024],
    noise_levels=[0.005, 0.01, 0.02] if FULL else [0.005],
    zne_scales=[1, 3, 5],
    noise_n=5,
    noise_windows=8 if FULL else 3,
    ibm_backend="ibm_marrakesh",
    declared_primary_backend="ibm_fez",
    hardware_validation_role="independent replication and artifact recovery",
    hw_n_qubits=7,
    hw_p=2,
    hw_windows=3,
    shots=4096,
    run_garch=True,
    run_lstm=FULL,
    bootstrap_repetitions=2000 if FULL else 300,
    bootstrap_block_length=22,
    amplitude_levels=[0.0, 0.001, 0.005, 0.01, 0.02] if FULL else [0.0, 0.005],
)
print("Run profile:", RUN_PROFILE)
print("Headline reservoir:", CONFIG["n_qubits"], "qubits; short", CONFIG["short"], "long", CONFIG["long"])print("Bootstrap:", CONFIG["bootstrap_repetitions"], "repetitions; block length", CONFIG["bootstrap_block_length"])
print("Amplitude damping levels:", CONFIG["amplitude_levels"])


## 3. Data — committed CSV first, public-source rebuild second

The locked headline feature set uses daily/weekly/monthly log realized variance, one-day log-RV change, and a VIX-implied volatility spread. NFCI is preserved in the committed market file for provenance and extensions but is not part of the locked five-feature headline vector.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def garman_klass(px):
    o, h, l, c = np.log(px["Open"]), np.log(px["High"]), np.log(px["Low"]), np.log(px["Close"])
    return (0.5 * (h - l) ** 2 - (2 * np.log(2) - 1) * (c - o) ** 2).clip(lower=1e-10)


def make_synthetic(T=3800, seed=SEED):
    rng = np.random.default_rng(seed)
    regime = np.zeros(T, dtype=int)
    state = 0
    for t in range(1, T):
        state = 1 - state if rng.random() < (0.02 if state == 0 else 0.05) else state
        regime[t] = state
    mu = np.where(regime == 0, np.log(7e-5), np.log(3e-4))
    x = np.empty(T)
    x[0] = mu[0]
    for t in range(1, T):
        x[t] = 0.94 * x[t-1] + 0.06 * mu[t] + 0.12 * rng.standard_normal()
    rv = np.exp(x)
    ret = np.sqrt(rv) * rng.standard_normal(T)
    vix = 100 * np.sqrt(252 * rv) + 3 * rng.standard_normal(T)
    return pd.DataFrame(
        {"ret": ret, "rv": rv, "vix": vix, "nfci": np.nan},
        index=pd.bdate_range("2010-01-01", periods=T),
    )


def _flatten_yfinance(frame):
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    return frame


def _download_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    frame = pd.read_csv(url)
    frame.columns = ["Date", series_id]
    frame["Date"] = pd.to_datetime(frame["Date"])
    frame[series_id] = pd.to_numeric(frame[series_id], errors="coerce")
    return frame.set_index("Date")[series_id]


def download_and_cache(cfg):
    import yfinance as yf
    px = _flatten_yfinance(
        yf.download(
            "^GSPC", start="2010-01-01", end="2025-01-02",
            auto_adjust=False, progress=False
        )
    )
    if px.empty:
        raise RuntimeError("Yahoo Finance returned no S&P 500 data.")
    vix = _download_fred("VIXCLS")
    nfci = _download_fred("NFCI")
    frame = pd.DataFrame(index=px.index)
    frame["ret"] = np.log(px["Close"]).diff()
    frame["rv"] = garman_klass(px)
    frame["vix"] = vix.reindex(frame.index).ffill()
    frame["nfci"] = nfci.reindex(frame.index).ffill()
    frame = frame.dropna(subset=["ret", "rv", "vix"])
    frame.index.name = "Date"
    frame.to_csv(cfg["data_csv"])
    return frame


def load_data(cfg):
    if cfg["use_synthetic"]:
        print("WARNING: use_synthetic=True. This is a demo run and must not be submitted as the locked result.")
        return make_synthetic(), "SYNTHETIC (demo only)"
    path = Path(cfg["data_csv"])
    if path.exists():
        frame = pd.read_csv(path, index_col=0, parse_dates=True)
        return frame, f"committed CSV ({path})"
    if cfg["allow_download"]:
        print("Committed market_data.csv was not found; rebuilding it from Yahoo Finance and FRED.")
        frame = download_and_cache(cfg)
        return frame, "public-source rebuild (downloaded and cached)"
    raise FileNotFoundError("data/market_data.csv is missing and downloads are disabled.")


data, data_src = load_data(CONFIG)
required_columns = {"ret", "rv", "vix"}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Market data is missing required columns: {sorted(missing_columns)}")
data = data.sort_index()
if not np.isfinite(data[["ret", "rv", "vix"]].to_numpy()).all():
    raise ValueError("Non-finite values remain in required market-data columns.")

MARKET_DATA_SHA256 = sha256_file(CONFIG["data_csv"]) if Path(CONFIG["data_csv"]).exists() else None
print(f"Source: {data_src}")
print(f"Rows: {len(data):,} | {data.index[0].date()} to {data.index[-1].date()}")
print("SHA256:", MARKET_DATA_SHA256)
display(data.head())

## 4. Preprocessing — chronological split and HAR-informed features

In [ ]:
FEATURE_NAMES = ["log_rv_daily", "log_rv_weekly", "log_rv_monthly", "delta_log_rv", "vix_spread"]


def build_features(frame, cfg):
    log_rv = np.log(frame["rv"].to_numpy())
    vix_spread = frame["vix"].to_numpy() / 100.0 - np.sqrt(252 * frame["rv"].to_numpy())
    L = cfg["window"]
    features, target, source_index = [], [], []
    for t in range(L, len(log_rv) - 1):
        window = log_rv[t-L+1:t+1]
        features.append([
            log_rv[t], window[-5:].mean(), window.mean(),
            log_rv[t] - log_rv[t-1], vix_spread[t]
        ])
        target.append(log_rv[t+1])
        source_index.append(t)
    return np.asarray(features), np.asarray(target), np.asarray(source_index)


F, y, idx = build_features(data, CONFIG)
ntr = int(CONFIG["train_frac"] * len(y))
encoder = MinMaxScaler((CONFIG["enc_low"], CONFIG["enc_high"])).fit(F[:ntr])
Fs = encoder.transform(F)
q1, q2 = np.quantile(np.exp(y[:ntr]), [1/3, 2/3])


def to_regime(values):
    return np.where(values < q1, 0, np.where(values < q2, 1, 2))


yt = y[ntr:]
test_idx = np.arange(ntr, len(y))
print(f"Samples: {len(y):,} | train: {ntr:,} | test: {len(yt):,} | features/day: {F.shape[1]}")
print("Feature order:", FEATURE_NAMES)

## 5. Dual-timescale transverse-field Ising reservoirs

- Short reservoir: $J=0.3$, $h=1.0$, $p=2$.
- Long reservoir: $J=1.2$, $h=0.4$, $p=6$.
- Angle encoding with feature re-uploading.

In [ ]:
def reservoir_circuit(x, n, J, h, p, dt=1.0, enc_frac=1.0):
    qc = QuantumCircuit(n)
    n_encoded = max(1, int(round(enc_frac * n)))
    for _ in range(p):
        for q in range(n_encoded):
            qc.ry(float(x[q % len(x)]), q)
        for q in range(n - 1):
            qc.rzz(2 * J * dt, q, q + 1)
        for q in range(n):
            qc.rx(2 * h * dt, q)
    return qc


short_example = reservoir_circuit(Fs[0], CONFIG["n_qubits"], **CONFIG["short"], dt=CONFIG["dt"])
long_example = reservoir_circuit(Fs[0], CONFIG["n_qubits"], **CONFIG["long"], dt=CONFIG["dt"])
print(f"Short: {short_example.num_qubits} qubits, depth {short_example.depth()}")
print(f"Long:  {long_example.num_qubits} qubits, depth {long_example.depth()}")

## 6. Hardware-native Pauli readout, optional entanglement features, and fading-memory feedback

In [ ]:
def z_ops(n):
    operators = []
    for i in range(n):
        label = ["I"] * n
        label[i] = "Z"
        operators.append(SparsePauliOp("".join(label[::-1])))
    for i in range(n - 1):
        label = ["I"] * n
        label[i] = "Z"
        label[i + 1] = "Z"
        operators.append(SparsePauliOp("".join(label[::-1])))
    return operators


def pauli_features(statevector, n):
    return [np.real(statevector.expectation_value(op)) for op in z_ops(n)]


def ent_features(statevector, n, keep):
    rho = partial_trace(statevector, list(range(n // 2, n)))
    eigenvalues = np.clip(np.real(np.linalg.eigvalsh(rho.data)), 1e-12, None)
    eigenvalues = np.sort(eigenvalues)[::-1][:keep]
    spectrum = -np.log(eigenvalues)
    if len(spectrum) < keep:
        spectrum = np.pad(spectrum, (0, keep - len(spectrum)), constant_values=spectrum[-1])
    return list(spectrum)


def reservoir_features(F_scaled, n, reservoir, cfg, feedback=None, pauli_only=True, enc_frac=1.0):
    use_feedback = cfg["feedback"] if feedback is None else feedback
    memory = np.zeros(cfg["mem_dim"])
    output = []
    for row in F_scaled:
        encoded = np.concatenate([row, memory]) if use_feedback else row
        circuit = reservoir_circuit(
            encoded, n, reservoir["J"], reservoir["h"], reservoir["p"],
            cfg["dt"], enc_frac
        )
        statevector = Statevector(circuit)
        pauli = pauli_features(statevector, n)
        output.append(pauli if pauli_only else pauli + ent_features(statevector, n, cfg["ent_keep"]))
        if use_feedback:
            summary = np.array([
                np.mean(pauli[:n]), np.mean(pauli[n:]), np.std(pauli[:n])
            ])[:cfg["mem_dim"]]
            memory = (1 - cfg["gamma"]) * memory + cfg["gamma"] * summary
    return np.asarray(output)

## 7. Build or validate the headline feature cache

In [ ]:
def array_digest(array):
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(contiguous.view(np.uint8)).hexdigest()


feature_signature = {
    "Fs_sha256": array_digest(Fs),
    "n_qubits": CONFIG["n_qubits"],
    "short": CONFIG["short"],
    "long": CONFIG["long"],
    "feedback": CONFIG["feedback"],
    "gamma": CONFIG["gamma"],
    "mem_dim": CONFIG["mem_dim"],
    "pauli_only": CONFIG["headline_pauli_only"],
}
cache_path = Path(CONFIG["features_cache"])
meta_path = Path(CONFIG["features_meta"])
use_cache = False
if cache_path.exists() and meta_path.exists() and not CONFIG["force_rebuild_features"]:
    try:
        previous_signature = json.loads(meta_path.read_text())
        cached = np.load(cache_path)
        expected_width = 2 * (2 * CONFIG["n_qubits"] - 1)
        use_cache = previous_signature == feature_signature and cached.shape == (len(Fs), expected_width)
    except Exception:
        use_cache = False

if use_cache:
    Q = cached
    print(f"Loaded validated feature cache: {Q.shape}")
else:
    start = time.time()
    Q = np.hstack([
        reservoir_features(Fs, CONFIG["n_qubits"], CONFIG["short"], CONFIG, pauli_only=True),
        reservoir_features(Fs, CONFIG["n_qubits"], CONFIG["long"], CONFIG, pauli_only=True),
    ])
    np.save(cache_path, Q)
    meta_path.write_text(json.dumps(feature_signature, indent=2, sort_keys=True))
    print(f"Built and cached headline features: {Q.shape} in {time.time() - start:.1f} seconds")

## 8. Metrics and train-only calibration

In [ ]:
def rmse(actual_log, forecast_log):
    return float(np.sqrt(np.mean((np.exp(actual_log) - np.exp(forecast_log)) ** 2)))


def qlike(actual_log, forecast_log):
    actual = np.exp(actual_log)
    forecast = np.clip(np.exp(forecast_log), 1e-12, None)
    return float(np.mean(actual / forecast - np.log(actual / forecast) - 1))


def regime_acc(actual_log, forecast_log):
    return float(np.mean(to_regime(np.exp(actual_log)) == to_regime(np.exp(forecast_log))))


def vol_sharpe(forecast_log, target_indices):
    returns_next = data["ret"].to_numpy()[idx[target_indices] + 1]
    position = np.clip(np.median(np.exp(forecast_log)) / np.exp(forecast_log), 0, 5)
    strategy = position * returns_next
    return float(np.mean(strategy) / (np.std(strategy) + 1e-12) * np.sqrt(252))


def mincer_zarnowitz(actual_log, forecast_log):
    X = sm.add_constant(np.exp(forecast_log))
    fit = sm.OLS(np.exp(actual_log), X).fit()
    joint_test = fit.f_test((np.eye(2), np.array([0, 1])))
    return float(fit.params[0]), float(fit.params[1]), float(joint_test.pvalue)


def bias_correct(y_train_true, pred_train, pred_test):
    intercept, slope = sm.OLS(
        np.exp(y_train_true), sm.add_constant(np.exp(pred_train))
    ).fit().params
    corrected = np.log(np.clip(intercept + slope * np.exp(pred_test), 1e-12, None))
    return corrected, (float(intercept), float(slope))

## 9. Train the QRC ridge head and produce headline forecasts

In [ ]:
qrc_model = make_pipeline(StandardScaler(), RidgeCV(alphas=CONFIG["ridge_alphas"]))
qrc_model.fit(Q[:ntr], y[:ntr])
yq = qrc_model.predict(Q[ntr:])
yq_train = qrc_model.predict(Q[:ntr])
yq_cal, (cal_intercept, cal_slope) = (
    bias_correct(y[:ntr], yq_train, yq) if CONFIG["bias_correction"] else (yq, (0.0, 1.0))
)
headline_mz_intercept, headline_mz_slope, headline_mz_p = mincer_zarnowitz(yt, yq)

print(f"Ridge alpha: {qrc_model[-1].alpha_:.4g}")
print(f"Dual+Pauli+FB          RMSE {rmse(yt, yq):.3e} | QLIKE {qlike(yt, yq):.4f} | Reg {regime_acc(yt, yq)*100:.1f}% | Sharpe {vol_sharpe(yq, test_idx):.3f}")
print(f"Dual+Pauli+FB calibrated RMSE {rmse(yt, yq_cal):.3e} | QLIKE {qlike(yt, yq_cal):.4f} | Reg {regime_acc(yt, yq_cal)*100:.1f}% | Sharpe {vol_sharpe(yq_cal, test_idx):.3f}")
print(f"Mincer-Zarnowitz headline: intercept {headline_mz_intercept:.2e}, slope {headline_mz_slope:.3f}, joint p {headline_mz_p:.3f}")

## 9b. Calm/turbulent regime head

In [ ]:
from sklearn.linear_model import LogisticRegression

regime_threshold = np.median(np.exp(y[:ntr]))
regime_binary = (np.exp(y) > regime_threshold).astype(int)
regime_scaler = StandardScaler().fit(Q[:ntr])
Q_regime = regime_scaler.transform(Q)
regime_head = LogisticRegression(max_iter=3000, random_state=SEED)
regime_head.fit(Q_regime[:ntr], regime_binary[:ntr])
regime2_acc = float(np.mean(regime_head.predict(Q_regime[ntr:]) == regime_binary[ntr:]))
majority_acc = float(max(np.mean(regime_binary[ntr:]), 1 - np.mean(regime_binary[ntr:])))
print(f"QRC calm/turbulent accuracy: {regime2_acc*100:.1f}%")
print(f"Majority baseline:            {majority_acc*100:.1f}%")

## 10. Classical baselines

In [ ]:
results = {}
prediction_store = {}


def log_result(name, source, prediction):
    prediction = np.asarray(prediction, dtype=float)
    prediction_store[name] = prediction.copy()
    _, mz_slope, mz_p = mincer_zarnowitz(yt, prediction)
    results[name] = dict(
        Source=source,
        RMSE=rmse(yt, prediction),
        QLIKE=qlike(yt, prediction),
        MZ_slope=mz_slope,
        MZ_p=mz_p,
        Regime_3state=regime_acc(yt, prediction),
        Sharpe=vol_sharpe(prediction, test_idx),
    )


log_result("QRC Dual+Pauli+FB", "QRC", yq)
if CONFIG["bias_correction"]:
    log_result("QRC Dual+Pauli+FB (calibrated)", "QRC", yq_cal)

# HAR-RV
X_har = np.column_stack([np.ones(len(y)), F[:, 0], F[:, 1], F[:, 2]])
beta_har = np.linalg.lstsq(X_har[:ntr], y[:ntr], rcond=None)[0]
log_result("HAR-RV", "Classical", X_har[ntr:] @ beta_har)

# Persistence
log_result("Persistence", "Classical", F[ntr:, 0])

# Echo State Network

def esn(input_series, n_train, size=200, spectral_radius=0.9, seed=SEED):
    rng = np.random.default_rng(seed)
    W_in = rng.uniform(-0.5, 0.5, (size, 1))
    W = rng.uniform(-0.5, 0.5, (size, size))
    W *= spectral_radius / np.max(np.abs(np.linalg.eigvals(W)))
    states = np.zeros((len(input_series), size))
    state = np.zeros(size)
    for t in range(len(input_series)):
        state = np.tanh(W_in[:, 0] * input_series[t] + W @ state)
        states[t] = state
    from sklearn.linear_model import Ridge
    return Ridge(alpha=1.0).fit(states[:n_train], y[:n_train]).predict(states[n_train:])


log_result("ESN", "Classical", esn(F[:, 0], ntr))

# LSTM with early stopping
if CONFIG["run_lstm"]:
    try:
        try:
            import torch
            import torch.nn as nn
        except Exception:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch"], check=True)
            import torch
            import torch.nn as nn
        torch.manual_seed(SEED)
        lookback = 10
        x_scaler = StandardScaler().fit(F[:ntr])
        X_scaled = x_scaler.transform(F)
        y_scaler = StandardScaler().fit(y[:ntr].reshape(-1, 1))
        Y_scaled = y_scaler.transform(y.reshape(-1, 1)).ravel()
        X_seq = np.asarray([X_scaled[t-lookback:t] for t in range(lookback, len(X_scaled))])
        Y_seq = Y_scaled[lookback:]
        train_indices = np.arange(0, ntr - lookback)
        test_indices = np.arange(ntr - lookback, len(Y_seq))
        validation_cut = int(0.85 * len(train_indices))
        train_main, validation = train_indices[:validation_cut], train_indices[validation_cut:]
        tensor = lambda a: torch.tensor(a, dtype=torch.float32)
        Xt, Yt = tensor(X_seq[train_main]), tensor(Y_seq[train_main]).view(-1, 1)
        Xv, Yv = tensor(X_seq[validation]), tensor(Y_seq[validation]).view(-1, 1)
        Xe = tensor(X_seq[test_indices])

        class LSTMModel(nn.Module):
            def __init__(self, input_dim, hidden=64):
                super().__init__()
                self.lstm = nn.LSTM(input_dim, hidden, batch_first=True)
                self.dropout = nn.Dropout(0.1)
                self.output = nn.Linear(hidden, 1)

            def forward(self, x):
                sequence, _ = self.lstm(x)
                return self.output(self.dropout(sequence[:, -1, :]))

        model = LSTMModel(F.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
        loss_fn = nn.MSELoss()
        batch_size = 128
        best_loss, bad_epochs, best_state = float("inf"), 0, None
        for epoch in range(600):
            model.train()
            permutation = torch.randperm(len(Xt))
            for start in range(0, len(Xt), batch_size):
                batch = permutation[start:start+batch_size]
                optimizer.zero_grad()
                loss_fn(model(Xt[batch]), Yt[batch]).backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                validation_loss = loss_fn(model(Xv), Yv).item()
            if validation_loss < best_loss - 1e-5:
                best_loss, bad_epochs = validation_loss, 0
                best_state = {key: value.clone() for key, value in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= 40:
                    break
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            prediction_scaled = model(Xe).numpy().ravel()
        prediction = y_scaler.inverse_transform(prediction_scaled.reshape(-1, 1)).ravel()
        log_result("LSTM", "Classical", prediction)
        print(f"LSTM trained with early stopping at epoch {epoch}.")
    except Exception as exc:
        print("LSTM skipped:", type(exc).__name__, exc)

# Static multi-step GARCH benchmark
if CONFIG["run_garch"]:
    try:
        from arch import arch_model
        fitted_garch = arch_model(data["ret"].to_numpy()[idx[:ntr]] * 100, vol="GARCH", p=1, q=1).fit(disp="off")
        variance_path = fitted_garch.forecast(horizon=len(yt), reindex=False).variance.values[-1] / 1e4
        log_result("GARCH(1,1)", "Classical", np.log(np.clip(variance_path, 1e-12, None)))
    except Exception as exc:
        print("GARCH skipped:", type(exc).__name__, exc)

print("Completed models:", list(results))

## 10b. Transition-event evaluation

Global accuracy can hide performance at the moments that matter most. We therefore evaluate the dedicated calm/turbulent head around each true regime change. A predicted event is a change in the predicted state. Event recall asks whether at least one predicted change occurs within the specified tolerance window; event precision penalizes unmatched alarms. We also report daily accuracy, balanced accuracy, and macro-F1 restricted to the union of transition windows.


In [ ]:
actual_state = regime_binary[ntr:]
predicted_state = regime_head.predict(Q_regime[ntr:])
majority_state = np.full_like(actual_state, int(np.mean(actual_state) >= 0.5))

true_events = np.flatnonzero(np.diff(actual_state) != 0) + 1
predicted_events = np.flatnonzero(np.diff(predicted_state) != 0) + 1


def event_scores(true_idx, pred_idx, tolerance):
    true_idx = np.asarray(true_idx, dtype=int)
    pred_idx = np.asarray(pred_idx, dtype=int)
    true_detected = sum(np.any(np.abs(pred_idx - event) <= tolerance) for event in true_idx)
    pred_matched = sum(np.any(np.abs(true_idx - event) <= tolerance) for event in pred_idx)
    recall = true_detected / len(true_idx) if len(true_idx) else np.nan
    precision = pred_matched / len(pred_idx) if len(pred_idx) else np.nan
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1


def transition_mask(length, events, tolerance):
    mask = np.zeros(length, dtype=bool)
    for event in events:
        lo = max(0, event - tolerance)
        hi = min(length, event + tolerance + 1)
        mask[lo:hi] = True
    return mask

transition_rows = []
for tolerance in [1, 3]:
    mask = transition_mask(len(actual_state), true_events, tolerance)
    event_precision, event_recall, event_f1 = event_scores(true_events, predicted_events, tolerance)
    transition_rows.append(dict(
        tolerance_days=tolerance,
        true_transition_events=int(len(true_events)),
        predicted_transition_events=int(len(predicted_events)),
        event_precision=float(event_precision),
        event_recall=float(event_recall),
        event_f1=float(event_f1),
        transition_window_days=int(mask.sum()),
        qrc_window_accuracy=float(np.mean(predicted_state[mask] == actual_state[mask])),
        qrc_window_balanced_accuracy=float(balanced_accuracy_score(actual_state[mask], predicted_state[mask])),
        qrc_window_macro_f1=float(f1_score(actual_state[mask], predicted_state[mask], average="macro", zero_division=0)),
        majority_window_accuracy=float(np.mean(majority_state[mask] == actual_state[mask])),
    ))

transition_metrics = pd.DataFrame(transition_rows)
transition_confusion = pd.DataFrame(
    confusion_matrix(actual_state, predicted_state, labels=[0, 1]),
    index=["actual_calm", "actual_turbulent"],
    columns=["predicted_calm", "predicted_turbulent"],
)

display(transition_metrics)
display(transition_confusion)


## 10c. Forecast significance and blocked uncertainty

The headline ranking is supplemented by pointwise loss comparisons. The Diebold-Mariano statistic uses a Newey-West long-run variance estimate. Moving-block bootstrap intervals preserve short-range dependence and report the QRC-minus-baseline difference; negative differences favor QRC.


In [ ]:
forecast_dates = pd.to_datetime(data.index[idx[test_idx] + 1])
forecast_predictions = pd.DataFrame({
    "Date": forecast_dates,
    "actual_log_variance": yt,
    "actual_variance": np.exp(yt),
})
for model_name, prediction in prediction_store.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", model_name).strip("_").lower()
    forecast_predictions[f"{safe_name}_log"] = prediction
    forecast_predictions[f"{safe_name}_variance"] = np.exp(prediction)


def pointwise_qlike(actual_log, forecast_log):
    actual = np.exp(actual_log)
    forecast = np.clip(np.exp(forecast_log), 1e-12, None)
    return actual / forecast - np.log(actual / forecast) - 1


def newey_west_variance(series, lag=None):
    values = np.asarray(series, dtype=float)
    values = values - values.mean()
    n = len(values)
    if lag is None:
        lag = max(1, int(round(n ** (1 / 3))))
    gamma0 = float(values @ values / n)
    long_run = gamma0
    for k in range(1, min(lag, n - 1) + 1):
        weight = 1 - k / (lag + 1)
        gamma = float(values[k:] @ values[:-k] / n)
        long_run += 2 * weight * gamma
    return max(long_run, 1e-18), lag


def diebold_mariano(actual_log, forecast_a, forecast_b, loss="qlike"):
    if loss == "qlike":
        differential = pointwise_qlike(actual_log, forecast_a) - pointwise_qlike(actual_log, forecast_b)
    elif loss == "squared_error":
        actual = np.exp(actual_log)
        differential = (actual - np.exp(forecast_a)) ** 2 - (actual - np.exp(forecast_b)) ** 2
    else:
        raise ValueError("loss must be qlike or squared_error")
    long_run, lag = newey_west_variance(differential)
    statistic = float(np.mean(differential) / np.sqrt(long_run / len(differential)))
    p_value = float(2 * stats.norm.sf(abs(statistic)))
    return statistic, p_value, float(np.mean(differential)), lag


def moving_block_indices(n, block_length, rng):
    starts = rng.integers(0, n - block_length + 1, size=int(np.ceil(n / block_length)))
    return np.concatenate([np.arange(start, start + block_length) for start in starts])[:n]


def bootstrap_difference(actual_log, forecast_a, forecast_b, metric, repetitions, block_length, seed=SEED):
    rng = np.random.default_rng(seed)
    differences = np.empty(repetitions)
    for b in range(repetitions):
        sample = moving_block_indices(len(actual_log), block_length, rng)
        if metric == "rmse":
            differences[b] = rmse(actual_log[sample], forecast_a[sample]) - rmse(actual_log[sample], forecast_b[sample])
        elif metric == "qlike":
            differences[b] = qlike(actual_log[sample], forecast_a[sample]) - qlike(actual_log[sample], forecast_b[sample])
        else:
            raise ValueError("metric must be rmse or qlike")
    low, high = np.quantile(differences, [0.025, 0.975])
    return float(np.mean(differences)), float(low), float(high)

qrc_candidate = prediction_store["QRC Dual+Pauli+FB (calibrated)"]
significance_rows = []
for benchmark in ["HAR-RV", "ESN", "LSTM", "Persistence"]:
    if benchmark not in prediction_store:
        continue
    comparator = prediction_store[benchmark]
    for loss in ["squared_error", "qlike"]:
        dm_stat, dm_p, mean_diff, nw_lag = diebold_mariano(yt, qrc_candidate, comparator, loss=loss)
        significance_rows.append(dict(
            comparison=f"QRC calibrated vs {benchmark}",
            analysis="Diebold-Mariano",
            metric=loss,
            estimate=mean_diff,
            ci_low=np.nan,
            ci_high=np.nan,
            statistic=dm_stat,
            p_value=dm_p,
            detail=f"Newey-West lag {nw_lag}; negative loss differential favors QRC",
        ))
    for metric in ["rmse", "qlike"]:
        estimate, ci_low, ci_high = bootstrap_difference(
            yt, qrc_candidate, comparator, metric,
            CONFIG["bootstrap_repetitions"], CONFIG["bootstrap_block_length"],
            seed=SEED + len(significance_rows),
        )
        significance_rows.append(dict(
            comparison=f"QRC calibrated vs {benchmark}",
            analysis="Moving-block bootstrap",
            metric=metric,
            estimate=estimate,
            ci_low=ci_low,
            ci_high=ci_high,
            statistic=np.nan,
            p_value=np.nan,
            detail=f"95% interval; block length {CONFIG['bootstrap_block_length']}; negative favors QRC",
        ))

forecast_significance = pd.DataFrame(significance_rows)
display(forecast_significance)


## 11. Headline results table

In [ ]:
headline_table = pd.DataFrame(results).T[
    ["Source", "RMSE", "QLIKE", "MZ_slope", "MZ_p", "Regime_3state", "Sharpe"]
].sort_values("RMSE")
display_table = headline_table.copy()
display_table["RMSE"] = display_table["RMSE"].map(lambda value: f"{value:.3e}")
display_table["QLIKE"] = display_table["QLIKE"].map(lambda value: f"{value:.4f}")
display_table["MZ_slope"] = display_table["MZ_slope"].map(lambda value: f"{value:.3f}")
display_table["MZ_p"] = display_table["MZ_p"].map(lambda value: f"{value:.3f}")
display_table["Regime_3state"] = (display_table["Regime_3state"] * 100).map(lambda value: f"{value:.1f}%")
display_table["Sharpe"] = display_table["Sharpe"].map(lambda value: f"{value:.3f}")
print(f"Data: {data_src} | test days: {len(yt)}")
display(display_table)

## 11b. Authentic Oxford-Man 5-minute realized-variance validation

In [ ]:
oxford_table = None
oxford_path = Path(CONFIG["oxford_csv"])
OXFORD_DATA_SHA256 = None
if oxford_path.exists():
    OXFORD_DATA_SHA256 = sha256_file(oxford_path)
    oxford_raw = pd.read_csv(oxford_path)["rv"].dropna().to_numpy()
    if np.median(oxford_raw) > 1e-3:
        oxford_raw = oxford_raw ** 2
        print("Oxford-Man input detected as realized volatility and squared to variance.")
    log_rv = np.log(np.clip(oxford_raw, 1e-12, None))
    L = CONFIG["window"]
    F_oxford, y_oxford = [], []
    for t in range(L, len(log_rv) - 1):
        window = log_rv[t-L+1:t+1]
        F_oxford.append([log_rv[t], window[-5:].mean(), window.mean(), log_rv[t] - log_rv[t-1]])
        y_oxford.append(log_rv[t+1])
    F_oxford = np.asarray(F_oxford)
    y_oxford = np.asarray(y_oxford)
    ntr_oxford = int(CONFIG["train_frac"] * len(y_oxford))
    yt_oxford = y_oxford[ntr_oxford:]
    F_scaled_oxford = MinMaxScaler((CONFIG["enc_low"], CONFIG["enc_high"])).fit(
        F_oxford[:ntr_oxford]
    ).transform(F_oxford)
    start = time.time()
    Q_oxford = np.hstack([
        reservoir_features(F_scaled_oxford, CONFIG["n_qubits"], CONFIG["short"], CONFIG, pauli_only=True),
        reservoir_features(F_scaled_oxford, CONFIG["n_qubits"], CONFIG["long"], CONFIG, pauli_only=True),
    ])
    model_oxford = make_pipeline(StandardScaler(), RidgeCV(alphas=CONFIG["ridge_alphas"]))
    model_oxford.fit(Q_oxford[:ntr_oxford], y_oxford[:ntr_oxford])
    qrc_oxford = model_oxford.predict(Q_oxford[ntr_oxford:])
    X_har_oxford = np.column_stack([
        np.ones(len(y_oxford)), F_oxford[:, 0], F_oxford[:, 1], F_oxford[:, 2]
    ])
    beta_oxford = np.linalg.lstsq(
        X_har_oxford[:ntr_oxford], y_oxford[:ntr_oxford], rcond=None
    )[0]
    oxford_predictions = {
        "QRC Dual+Pauli+FB": qrc_oxford,
        "HAR-RV": X_har_oxford[ntr_oxford:] @ beta_oxford,
        "Persistence": F_oxford[ntr_oxford:, 0],
    }
    oxford_table = pd.DataFrame([
        dict(
            Model=name,
            RMSE=rmse(yt_oxford, prediction),
            QLIKE=qlike(yt_oxford, prediction),
            MZ_slope=mincer_zarnowitz(yt_oxford, prediction)[1],
        )
        for name, prediction in oxford_predictions.items()
    ])
    print(f"Oxford-Man test days: {len(yt_oxford)} | feature build: {time.time() - start:.1f} seconds")
    display(oxford_table.assign(
        RMSE=lambda d: d.RMSE.map("{:.3e}".format),
        QLIKE=lambda d: d.QLIKE.map("{:.4f}".format),
        MZ_slope=lambda d: d.MZ_slope.map("{:.3f}".format),
    ))
    print("Interpretation: QRC is competitive with HAR-RV on genuine 5-minute realized variance; this is not claimed as a statistically significant win.")
else:
    print("Oxford-Man validation skipped because data/oxman_spx.csv is absent. Include the committed mirror CSV in the judge bundle.")

In [ ]:
from pathlib import Path

Path("results").mkdir(parents=True, exist_ok=True)

if oxford_table is None:
    raise RuntimeError(
        "Oxford-Man results are not currently in memory. "
        "Rerun the Oxford-Man validation cell first."
    )

output_file = Path("results/oxford_man_metrics.csv")
oxford_table.to_csv(output_file, index=False)

print("Saved:", output_file.resolve())
print("File exists:", output_file.exists())
print("File size:", output_file.stat().st_size, "bytes")
display(oxford_table)

---
# Phase 3 execution studies

In [ ]:
S = CONFIG["study_sample"] or len(y)
Fs_study, y_study = Fs[-S:], y[-S:]
ntr_study = int(CONFIG["train_frac"] * len(y_study))
yt_study = y_study[ntr_study:]


def evaluate_features(features):
    model = make_pipeline(StandardScaler(), RidgeCV(alphas=CONFIG["ridge_alphas"]))
    model.fit(features[:ntr_study], y_study[:ntr_study])
    prediction = model.predict(features[ntr_study:])
    return rmse(yt_study, prediction), qlike(yt_study, prediction)


HAR_RMSE = results["HAR-RV"]["RMSE"]
ESN_RMSE = results["ESN"]["RMSE"]
print(f"Study rows: {S:,} | train: {ntr_study:,} | test: {len(yt_study):,}")

## 12. Ablation

In [ ]:
ablation = {}


def build_variant(n_qubits, reservoir_choice, entanglement, feedback):
    reservoirs = [CONFIG["short"], CONFIG["long"]] if reservoir_choice == "dual" else [CONFIG["long"]]
    return np.hstack([
        reservoir_features(
            Fs_study, n_qubits, reservoir, CONFIG,
            feedback=feedback, pauli_only=not entanglement
        )
        for reservoir in reservoirs
    ])


variants = {
    "Dual Pauli +FB": ("dual", False, True),
    "Dual +Ent +FB": ("dual", True, True),
    "Dual +Ent -FB": ("dual", True, False),
    "Single +Ent +FB": ("single", True, True),
    "Single +Ent -FB": ("single", True, False),
    "Single Pauli -FB": ("single", False, False),
}
for name, (choice, entanglement, feedback) in variants.items():
    variant_features = build_variant(CONFIG["n_qubits"], choice, entanglement, feedback)
    variant_rmse, variant_qlike = evaluate_features(variant_features)
    ablation[name] = dict(RMSE=variant_rmse, QLIKE=variant_qlike)

ablation_table = pd.DataFrame(ablation).T.sort_values("RMSE")
display(ablation_table.assign(
    RMSE=lambda d: d.RMSE.map("{:.3e}".format),
    QLIKE=lambda d: d.QLIKE.map("{:.4f}".format),
))

## 13. Reservoir-size scaling

In [ ]:
import itertools
scaling_rows = []
for n_qubits, depth_p in itertools.product(CONFIG["scan_n"], CONFIG["scan_p"]):
    start = time.time()
    features = reservoir_features(
        Fs_study, n_qubits, dict(J=0.3, h=1.0, p=depth_p), CONFIG
    )
    size_rmse, size_qlike = evaluate_features(features)
    scaling_rows.append(dict(
        n=n_qubits, p=depth_p, RMSE=size_rmse, QLIKE=size_qlike,
        seconds=round(time.time() - start, 1)
    ))
scaling_table = pd.DataFrame(scaling_rows)
display(scaling_table)

## 14. Encoding-density scaling

In [ ]:
encoding_rows = []
for fraction in CONFIG["enc_fracs"]:
    features = reservoir_features(
        Fs_study, CONFIG["n_qubits"], CONFIG["short"], CONFIG, enc_frac=fraction
    )
    density_rmse, density_qlike = evaluate_features(features)
    encoding_rows.append(dict(
        encoding_fraction=fraction,
        qubits_encoded=max(1, round(fraction * CONFIG["n_qubits"])),
        RMSE=density_rmse,
        QLIKE=density_qlike,
    ))
encoding_table = pd.DataFrame(encoding_rows)
display(encoding_table.assign(
    RMSE=lambda d: d.RMSE.map("{:.3e}".format),
    QLIKE=lambda d: d.QLIKE.map("{:.4f}".format),
))

## 15. Shot-budget scaling

In [ ]:
def pauli_shots(statevector, n, shots, rng):
    probabilities = np.abs(statevector.data) ** 2
    probabilities /= probabilities.sum()
    samples = rng.choice(len(probabilities), size=shots, p=probabilities)
    bits = (samples[:, None] >> np.arange(n)) & 1
    z = 1 - 2 * bits
    features = list(z.mean(axis=0))
    for i in range(n - 1):
        features.append(np.mean(z[:, i] * z[:, i + 1]))
    return features


def shot_features(F_scaled, n, reservoir, cfg, shots, seed=SEED):
    rng = np.random.default_rng(seed)
    memory = np.zeros(cfg["mem_dim"])
    output = []
    for row in F_scaled:
        encoded = np.concatenate([row, memory]) if cfg["feedback"] else row
        statevector = Statevector(
            reservoir_circuit(encoded, n, reservoir["J"], reservoir["h"], reservoir["p"], cfg["dt"])
        )
        pauli = pauli_shots(statevector, n, shots, rng) if shots else pauli_features(statevector, n)
        output.append(pauli)
        if cfg["feedback"]:
            summary = np.array([
                np.mean(pauli[:n]), np.mean(pauli[n:]), np.std(pauli[:n])
            ])[:cfg["mem_dim"]]
            memory = (1 - cfg["gamma"]) * memory + cfg["gamma"] * summary
    return np.asarray(output)


shot_rows = []
for shot_count in CONFIG["shot_list"] + [None]:
    features = shot_features(Fs_study, CONFIG["n_qubits"], CONFIG["short"], CONFIG, shot_count)
    shot_rmse, shot_qlike = evaluate_features(features)
    shot_rows.append(dict(
        shots=shot_count if shot_count is not None else "exact",
        RMSE=shot_rmse,
        QLIKE=shot_qlike,
    ))
shot_table = pd.DataFrame(shot_rows)
display(shot_table.assign(
    RMSE=lambda d: d.RMSE.map("{:.3e}".format),
    QLIKE=lambda d: d.QLIKE.map("{:.4f}".format),
))

## 16. Noise and zero-noise extrapolation

In [ ]:
def noisy_expectations(circuit, n, operators, one_qubit_error, two_qubit_error):
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(one_qubit_error, 1), ["rx", "ry"])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(two_qubit_error, 2), ["rzz"])
    simulator = AerSimulator(method="density_matrix", noise_model=noise_model)
    noisy_circuit = circuit.copy()
    noisy_circuit.save_density_matrix()
    density_matrix = simulator.run(noisy_circuit).result().data()["density_matrix"]
    return np.asarray([np.real(density_matrix.expectation_value(op)) for op in operators])


def fold_circuit(circuit, fold_count):
    folded = circuit.copy()
    for _ in range(fold_count):
        folded = folded.compose(circuit.inverse()).compose(circuit)
    return folded


noise_n = CONFIG["noise_n"]
noise_operators = z_ops(noise_n)
fold_counts = [(scale - 1) // 2 for scale in CONFIG["zne_scales"]]
noise_rows = []
for one_qubit_error in CONFIG["noise_levels"]:
    two_qubit_error = 2 * one_qubit_error
    raw_errors, mitigated_errors = [], []
    for row in Fs_study[:CONFIG["noise_windows"]]:
        circuit = reservoir_circuit(row, noise_n, 0.3, 1.0, 2)
        ideal = np.asarray([np.real(Statevector(circuit).expectation_value(op)) for op in noise_operators])
        noisy_values = [
            noisy_expectations(
                fold_circuit(circuit, fold_count), noise_n, noise_operators,
                one_qubit_error, two_qubit_error
            )
            for fold_count in fold_counts
        ]
        extrapolated = np.asarray([
            np.polyfit(CONFIG["zne_scales"], [values[j] for values in noisy_values], 1)[1]
            for j in range(len(noise_operators))
        ])
        raw_errors.append(np.mean(np.abs(noisy_values[0] - ideal)))
        mitigated_errors.append(np.mean(np.abs(extrapolated - ideal)))
    raw_mae = float(np.mean(raw_errors))
    zne_mae = float(np.mean(mitigated_errors))
    noise_rows.append(dict(
        depol_1q=one_qubit_error,
        depol_2q=two_qubit_error,
        raw_mae=raw_mae,
        zne_mae=zne_mae,
        recovered_percent=round(100 * (1 - zne_mae / raw_mae)),
    ))
noise_table = pd.DataFrame(noise_rows)
display(noise_table.assign(
    raw_mae=lambda d: d.raw_mae.map("{:.4f}".format),
    zne_mae=lambda d: d.zne_mae.map("{:.4f}".format),
))

## 16b. Amplitude damping and combined channel

The challenge explicitly requests amplitude damping in addition to depolarizing noise. This experiment applies amplitude damping after one-qubit gates and the tensor product of two damping channels after each `rzz` gate, while retaining a modest depolarizing background. The reported outcome is feature MAE against the ideal Pauli vector, averaged across representative financial windows.


In [ ]:
def combined_noise_expectations(circuit, n, operators, depol_1q, depol_2q, damping_gamma):
    model = NoiseModel()
    one_qubit = depolarizing_error(depol_1q, 1).compose(amplitude_damping_error(damping_gamma))
    damping_two = amplitude_damping_error(damping_gamma).tensor(amplitude_damping_error(damping_gamma))
    two_qubit = depolarizing_error(depol_2q, 2).compose(damping_two)
    model.add_all_qubit_quantum_error(one_qubit, ["rx", "ry"])
    model.add_all_qubit_quantum_error(two_qubit, ["rzz"])
    simulator = AerSimulator(method="density_matrix", noise_model=model)
    qc = circuit.copy()
    qc.save_density_matrix()
    density = simulator.run(qc).result().data()["density_matrix"]
    return np.asarray([np.real(density.expectation_value(op)) for op in operators])

amplitude_rows = []
background_depol_1q = 0.005
background_depol_2q = 0.010
for damping_gamma in CONFIG["amplitude_levels"]:
    window_errors = []
    for row in Fs_study[:CONFIG["noise_windows"]]:
        circuit = reservoir_circuit(row, noise_n, 0.3, 1.0, 2)
        ideal = np.asarray([np.real(Statevector(circuit).expectation_value(op)) for op in noise_operators])
        noisy = combined_noise_expectations(
            circuit, noise_n, noise_operators,
            background_depol_1q, background_depol_2q, damping_gamma,
        )
        window_errors.append(float(np.mean(np.abs(noisy - ideal))))
    amplitude_rows.append(dict(
        amplitude_gamma=damping_gamma,
        depol_1q=background_depol_1q,
        depol_2q=background_depol_2q,
        feature_mae=float(np.mean(window_errors)),
        windows=int(CONFIG["noise_windows"]),
        qubits=int(noise_n),
    ))

amplitude_damping_table = pd.DataFrame(amplitude_rows)
display(amplitude_damping_table)


## 17. IBM `ibm_marrakesh` replication and complete artifact capture

This cell is intentionally enabled for the team rerun.

- The declared primary Phase 3 hardware result remains the earlier IBM `ibm_fez` validation.
- This execution is an independent replication on IBM `ibm_marrakesh`.
- The new metrics must be reported exactly as produced.
- The notebook creates both canonical files required by the verifier and backend-specific copies for provenance.
- A job ledger is updated after each submission and each completed window, allowing recovery after a qBraid disconnect.
- Existing canonical hardware files are backed up before replacement.
- Credentials are requested through hidden prompts and are not written to the notebook or result files.


In [ ]:
RUN_HARDWARE_NOW = False
hardware_summary = None
hardware_observables = None


def rank_backends(service, min_qubits=1):
    rows = []
    for backend in service.backends(operational=True, simulator=False):
        try:
            status = backend.status()
            rows.append(dict(
                name=backend.name,
                qubits=backend.num_qubits,
                pending_jobs=status.pending_jobs,
                operational=getattr(status, "operational", None),
                status_msg=getattr(status, "status_msg", None),
            ))
        except Exception as exc:
            rows.append(dict(
                name=getattr(backend, "name", "unknown"),
                qubits=getattr(backend, "num_qubits", None),
                pending_jobs=np.inf,
                operational=None,
                status_msg=f"status unavailable: {exc}",
            ))
    frame = pd.DataFrame(rows)
    if len(frame):
        frame = frame[frame["qubits"] >= min_qubits].sort_values(
            ["pending_jobs", "name"]
        ).reset_index(drop=True)
    return frame


def connect_backend(service, cfg):
    requested = cfg["ibm_backend"]
    if requested and str(requested).lower() != "auto":
        backend = service.backend(requested)
        try:
            status = backend.status()
            print(
                f"Selected backend: {backend.name} | "
                f"qubits={backend.num_qubits} | "
                f"pending_jobs={status.pending_jobs} | "
                f"operational={getattr(status, 'operational', 'unknown')}"
            )
        except Exception as exc:
            print(f"Selected backend: {backend.name}; status lookup failed: {exc}")
        return backend

    candidates = rank_backends(service, cfg["hw_n_qubits"])
    if candidates.empty:
        raise RuntimeError("No eligible IBM backend is available for this account.")
    display(candidates)
    return service.backend(candidates.iloc[0]["name"])


def _backup_existing(path):
    import shutil
    path = Path(path)
    if not path.exists():
        return None
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    backup = path.with_name(f"{path.stem}_backup_{stamp}{path.suffix}")
    shutil.copy2(path, backup)
    print("Backed up existing artifact:", backup)
    return str(backup)


def _write_json(path, payload):
    Path(path).write_text(
        json.dumps(payload, indent=2, default=str),
        encoding="utf-8",
    )


def run_hardware_validation(cfg):
    import shutil
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    token = (
        os.environ.get("QISKIT_IBM_TOKEN")
        or getpass.getpass("IBM Quantum API key (hidden): ").strip()
    )
    instance = (
        os.environ.get("QISKIT_IBM_INSTANCE")
        or getpass.getpass(
            "IBM instance/CRN (hidden; blank if not required): "
        ).strip()
    )
    if not token:
        raise RuntimeError("No IBM API key was provided.")

    service_kwargs = dict(
        channel="ibm_quantum_platform",
        token=token,
    )
    if instance:
        service_kwargs["instance"] = instance

    service = QiskitRuntimeService(**service_kwargs)

    n = cfg["hw_n_qubits"]
    p = cfg["hw_p"]
    windows = cfg["hw_windows"]
    reservoir = cfg["short"]
    operators = z_ops(n)
    backend = connect_backend(service, cfg)
    backend_name = backend.name

    if backend_name != cfg["ibm_backend"]:
        raise RuntimeError(
            f"Connected to {backend_name}, but the configured backend is "
            f"{cfg['ibm_backend']}. No hardware jobs were submitted."
        )

    print(f"Submitting {windows} validation windows to {backend_name}...")

    pass_manager = generate_preset_pass_manager(
        optimization_level=3,
        backend=backend,
    )
    estimator = Estimator(mode=backend)
    estimator.options.default_shots = cfg["shots"]

    selected = test_idx[:windows]
    start = time.time()
    simulator_values = []
    hardware_values = []
    records = []
    job_ids = []
    transpiled_depths = []

    safe_backend = backend_name.replace("-", "_")
    ledger_path = Path(
        f"results/hardware_job_ledger_{safe_backend}.json"
    )
    backend_summary_path = Path(
        f"results/hardware_validation_{safe_backend}.json"
    )
    backend_observables_path = Path(
        f"results/hardware_observables_{safe_backend}.csv"
    )
    canonical_summary_path = Path("results/hardware_validation.json")
    canonical_observables_path = Path("results/hardware_observables.csv")

    ledger = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "backend": backend_name,
        "declared_primary_backend": cfg.get("declared_primary_backend"),
        "validation_role": cfg.get("hardware_validation_role"),
        "configured_windows": int(windows),
        "shots_per_circuit": int(cfg["shots"]),
        "status": "STARTED",
        "windows": [],
    }
    _write_json(ledger_path, ledger)
    print("Hardware recovery ledger:", ledger_path)

    for window_number, feature_index in enumerate(selected, start=1):
        circuit = reservoir_circuit(
            Fs[feature_index],
            n,
            reservoir["J"],
            reservoir["h"],
            p,
            cfg["dt"],
        )

        statevector = Statevector(circuit)
        simulated = [
            float(np.real(statevector.expectation_value(op)))
            for op in operators
        ]

        isa_circuit = pass_manager.run(circuit)
        transpiled_depth = int(isa_circuit.depth())
        transpiled_depths.append(transpiled_depth)
        isa_operators = [
            op.apply_layout(isa_circuit.layout)
            for op in operators
        ]

        submitted_utc = datetime.now(timezone.utc).isoformat()
        job = estimator.run([(isa_circuit, isa_operators)])
        job_id = job.job_id()
        job_ids.append(job_id)

        window_record = {
            "window": int(window_number),
            "feature_index": int(feature_index),
            "job_id": job_id,
            "submitted_utc": submitted_utc,
            "completed_utc": None,
            "status": "SUBMITTED",
            "transpiled_depth": transpiled_depth,
            "observable_count": int(len(operators)),
        }
        ledger["windows"].append(window_record)
        ledger["updated_utc"] = datetime.now(timezone.utc).isoformat()
        ledger["status"] = "IN_PROGRESS"
        _write_json(ledger_path, ledger)

        print(
            f"  window {window_number}/{windows} submitted | "
            f"job_id={job_id} | depth={transpiled_depth}"
        )

        try:
            result = job.result()
            measured = [
                float(value)
                for value in np.real(result[0].data.evs)
            ]
        except Exception as exc:
            window_record["status"] = "RESULT_ERROR"
            window_record["error"] = f"{type(exc).__name__}: {exc}"
            ledger["updated_utc"] = datetime.now(timezone.utc).isoformat()
            ledger["status"] = "PARTIAL"
            _write_json(ledger_path, ledger)
            raise

        if len(measured) != len(simulated):
            window_record["status"] = "OBSERVABLE_COUNT_MISMATCH"
            window_record["measured_count"] = int(len(measured))
            ledger["updated_utc"] = datetime.now(timezone.utc).isoformat()
            ledger["status"] = "PARTIAL"
            _write_json(ledger_path, ledger)
            raise RuntimeError(
                f"Window {window_number} returned {len(measured)} values; "
                f"{len(simulated)} were expected."
            )

        simulator_values.extend(simulated)
        hardware_values.extend(measured)

        for observable_index, (sim_value, hw_value) in enumerate(
            zip(simulated, measured)
        ):
            records.append(dict(
                backend=backend_name,
                validation_role=cfg.get("hardware_validation_role"),
                window=int(window_number),
                feature_index=int(feature_index),
                job_id=job_id,
                observable_index=int(observable_index),
                simulator=float(sim_value),
                hardware=float(hw_value),
                absolute_error=float(abs(sim_value - hw_value)),
            ))

        # Save partial observables after every completed window.
        pd.DataFrame(records).to_csv(
            backend_observables_path,
            index=False,
        )

        window_record["completed_utc"] = datetime.now(
            timezone.utc
        ).isoformat()
        window_record["status"] = "DONE"
        window_record["measured_count"] = int(len(measured))
        ledger["updated_utc"] = datetime.now(timezone.utc).isoformat()
        ledger["completed_windows"] = int(window_number)
        _write_json(ledger_path, ledger)

        print(
            f"  window {window_number}/{windows} complete "
            f"({time.time() - start:.0f} seconds elapsed)"
        )

    simulator_values = np.asarray(
        simulator_values,
        dtype=float,
    )
    hardware_values = np.asarray(
        hardware_values,
        dtype=float,
    )

    wall_seconds = time.time() - start
    correlation = float(
        np.corrcoef(simulator_values, hardware_values)[0, 1]
    )
    mae = float(
        np.mean(np.abs(simulator_values - hardware_values))
    )

    summary = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "backend": backend_name,
        "declared_primary_backend": cfg.get("declared_primary_backend"),
        "validation_role": cfg.get("hardware_validation_role"),
        "qubits": int(n),
        "reservoir_p": int(p),
        "transpiled_depth": int(transpiled_depths[0]),
        "transpiled_depths": transpiled_depths,
        "shots_per_circuit": int(cfg["shots"]),
        "windows": int(windows),
        "wall_seconds": float(wall_seconds),
        "observables": int(len(simulator_values)),
        "feature_correlation": correlation,
        "feature_mae": mae,
        "job_ids": job_ids,
        "job_ledger": str(ledger_path),
        "backend_specific_observables": str(
            backend_observables_path
        ),
    }

    # Save backend-specific artifacts first.
    _write_json(backend_summary_path, summary)
    hardware_frame = pd.DataFrame(records)
    hardware_frame.to_csv(
        backend_observables_path,
        index=False,
    )

    # Back up and then create canonical verifier-facing artifacts.
    canonical_backups = {
        "summary": _backup_existing(canonical_summary_path),
        "observables": _backup_existing(canonical_observables_path),
    }
    summary["canonical_backups"] = canonical_backups
    _write_json(backend_summary_path, summary)
    _write_json(canonical_summary_path, summary)
    hardware_frame.to_csv(
        canonical_observables_path,
        index=False,
    )

    ledger["updated_utc"] = datetime.now(timezone.utc).isoformat()
    ledger["status"] = "DONE"
    ledger["completed_windows"] = int(windows)
    ledger["feature_correlation"] = correlation
    ledger["feature_mae"] = mae
    ledger["canonical_summary"] = str(canonical_summary_path)
    ledger["canonical_observables"] = str(
        canonical_observables_path
    )
    _write_json(ledger_path, ledger)

    print("\nHardware validation completed.")
    print(json.dumps(summary, indent=2))
    print("Saved:", backend_summary_path)
    print("Saved:", backend_observables_path)
    print("Saved:", canonical_summary_path)
    print("Saved:", canonical_observables_path)
    print("Saved:", ledger_path)

    return summary, hardware_frame


if RUN_HARDWARE_NOW:
    try:
        hardware_summary, hardware_observables = (
            run_hardware_validation(CONFIG)
        )
    except Exception as exc:
        print(
            f"Hardware validation did not complete: "
            f"{type(exc).__name__}: {exc}"
        )
        print(
            "Do not resubmit blindly. Inspect "
            "results/hardware_job_ledger_ibm_marrakesh.json "
            "and recover any recorded job IDs."
        )
else:
    existing_hardware = Path(
        "results/hardware_validation.json"
    )
    if existing_hardware.exists():
        hardware_summary = json.loads(
            existing_hardware.read_text(
                encoding="utf-8"
            )
        )
        print(
            "Loaded existing hardware summary from "
            "results/hardware_validation.json"
        )
        print(json.dumps(hardware_summary, indent=2))
    else:
        print(
            "Hardware lane skipped. "
            "Set RUN_HARDWARE_NOW=True to execute it."
        )


## IBM job recovery — disabled for the Marrakesh top-to-bottom run

The former Fez recovery cell is retained as a non-executing reference only.  
Do not retrieve job `d9bobhu6hjac73fg01a0` during this run.

If the Marrakesh session disconnects, use the job IDs written to:

```text
results/hardware_job_ledger_ibm_marrakesh.json
```

Retrieve only the incomplete Marrakesh job rather than submitting a duplicate.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from qiskit_ibm_runtime import QiskitRuntimeService

import csv
import getpass
import json
import os


# ---------------------------------------------------------
# EXISTING IBM JOB — this does not submit another job
# ---------------------------------------------------------
IBM_JOB_ID = ""  # Populate only with a Marrakesh job ID from the new ledger
RECOVER_IBM_JOB = False

Path("results").mkdir(parents=True, exist_ok=True)


if RECOVER_IBM_JOB:

    token = os.environ.get("QISKIT_IBM_TOKEN")

    if not token:
        token = getpass.getpass(
            "IBM Quantum API key: "
        ).strip()

    instance = os.environ.get("QISKIT_IBM_INSTANCE")

    if not instance:
        instance = getpass.getpass(
            "IBM Quantum instance/CRN: "
        ).strip()

    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=token,
        instance=instance,
    )

    print("Retrieving existing IBM job...")
    print("Job ID:", IBM_JOB_ID)

    job = service.job(IBM_JOB_ID)
    status = str(job.status())

    print("Status:", status)

    try:
        backend = job.backend()
        backend_name = getattr(
            backend,
            "name",
            str(backend),
        )
    except Exception:
        backend_name = None

    metadata = {
        "retrieved_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "job_id": job.job_id(),
        "status": status,
        "backend": backend_name,
        "creation_date": str(job.creation_date),
        "primitive_id": getattr(
            job,
            "primitive_id",
            None,
        ),
    }

    try:
        metadata["metrics"] = job.metrics()
    except Exception as exc:
        metadata["metrics_note"] = str(exc)

    if status != "DONE":
        raise RuntimeError(
            f"Job is not complete. Current status: {status}. "
            "Do not submit it again."
        )

    job_result = job.result()

    records = []

    for pub_index, pub_result in enumerate(job_result):

        evs = pub_result.data.evs

        # Convert Qiskit/NumPy output to an ordinary Python list.
        if hasattr(evs, "tolist"):
            evs = evs.tolist()

        # Ensure one-dimensional processing.
        if not isinstance(evs, list):
            evs = [evs]

        flattened_values = []

        for item in evs:
            if isinstance(item, list):
                flattened_values.extend(item)
            else:
                flattened_values.append(item)

        print(
            f"Recovered {len(flattened_values)} "
            f"expectation values from pub {pub_index}."
        )

        for observable_index, value in enumerate(
            flattened_values
        ):
            records.append({
                "job_id": IBM_JOB_ID,
                "pub_index": pub_index,
                "observable_index": observable_index,
                "hardware_expectation_value": float(value),
            })

    csv_path = Path(
        f"results/recovered_ibm_{IBM_JOB_ID}_evs.csv"
    )

    with csv_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=[
                "job_id",
                "pub_index",
                "observable_index",
                "hardware_expectation_value",
            ],
        )

        writer.writeheader()
        writer.writerows(records)

    metadata["expectation_value_count"] = len(records)
    metadata["result_csv"] = str(csv_path)

    json_path = Path(
        f"results/recovered_ibm_{IBM_JOB_ID}_metadata.json"
    )

    json_path.write_text(
        json.dumps(
            metadata,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

    print("\nRecovery completed.")
    print("Saved:", csv_path)
    print("Saved:", json_path)
    print("Expectation values:", len(records))

else:
    print(
        "IBM recovery is disabled. "
        "Set IBM_JOB_ID to a recorded Marrakesh job ID and RECOVER_IBM_JOB=True only after a disconnect."
    )

## 18. Figures

In [ ]:
%pip install -q pandas matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Baseline RMSE
ordered_names = list(headline_table.sort_values("RMSE", ascending=False).index)
rmse_values = [headline_table.loc[name, "RMSE"] for name in ordered_names]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(ordered_names, rmse_values)
ax.set_title("QRC and classical baselines — RMSE")
ax.set_xlabel("RMSE (lower is better)")
ax.set_xlim(0, max(rmse_values) * 1.18)
for bar, value in zip(bars, rmse_values):
    ax.text(value, bar.get_y() + bar.get_height()/2, f" {value:.4g}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig("figures/fig1a_baselines_rmse.png", dpi=160)
plt.show()

# Baseline QLIKE
qlike_values = [headline_table.loc[name, "QLIKE"] for name in ordered_names]
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(ordered_names, qlike_values)
ax.set_title("QRC and classical baselines — QLIKE")
ax.set_xlabel("QLIKE (lower is better)")
ax.set_xlim(0, max(qlike_values) * 1.18)
for bar, value in zip(bars, qlike_values):
    ax.text(value, bar.get_y() + bar.get_height()/2, f" {value:.4f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig("figures/fig1b_baselines_qlike.png", dpi=160)
plt.show()

# Ablation
ablation_plot = ablation_table.sort_values("RMSE")
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(range(len(ablation_plot)), ablation_plot["RMSE"] * 1e5)
bars[0].set_hatch("//")
ax.axhline(HAR_RMSE * 1e5, linestyle="--", linewidth=0.9, label="HAR-RV")
ax.axhline(ESN_RMSE * 1e5, linestyle=":", linewidth=0.9, label="ESN")
ax.set_xticks(range(len(ablation_plot)))
ax.set_xticklabels(ablation_plot.index, rotation=18, ha="right", fontsize=8.5)
ax.set_ylabel("RMSE (×10⁻⁵)")
ax.set_title("Component ablation — hatched bar is best")
ax.legend()
fig.tight_layout()
fig.savefig("figures/fig2_ablation.png", dpi=160)
plt.show()

# Reservoir-size scaling
fig, ax = plt.subplots(figsize=(8, 4.5))
for depth_p in sorted(scaling_table.p.unique()):
    subset = scaling_table[scaling_table.p == depth_p].sort_values("n")
    ax.plot(subset.n, subset.RMSE * 1e5, marker="o", label=f"p={depth_p}")
ax.axhline(HAR_RMSE * 1e5, linestyle="--", linewidth=0.9, label="HAR-RV")
ax.set_xlabel("Qubits n")
ax.set_ylabel("RMSE (×10⁻⁵)")
ax.set_title("Reservoir-size scaling")
ax.legend()
fig.tight_layout()
fig.savefig("figures/fig3_scaling.png", dpi=160)
plt.show()

# Shot budget
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot([str(value) for value in shot_table.shots], shot_table.RMSE * 1e5, marker="s")
ax.axhline(HAR_RMSE * 1e5, linestyle="--", linewidth=0.9, label="HAR-RV")
ax.set_xlabel("Shots")
ax.set_ylabel("RMSE (×10⁻⁵)")
ax.set_title("Shot-budget scaling")
ax.legend()
fig.tight_layout()
fig.savefig("figures/fig4_shots.png", dpi=160)
plt.show()

# Noise and ZNE
positions = np.arange(len(noise_table))
width = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.bar(positions - width/2, noise_table.raw_mae, width, label="Raw noisy")
ax.bar(positions + width/2, noise_table.zne_mae, width, label="ZNE mitigated")
for i, row in noise_table.iterrows():
    ax.text(i + width/2, row.zne_mae, f"-{int(row.recovered_percent)}%", ha="center", va="bottom", fontsize=8)
ax.set_xticks(positions)
ax.set_xticklabels(noise_table.depol_1q)
ax.set_xlabel("One-qubit depolarizing rate")
ax.set_ylabel("Feature MAE vs ideal")
ax.set_title("Noise impact and ZNE recovery")
ax.legend()
fig.tight_layout()
fig.savefig("figures/fig5_noise_zne.png", dpi=160)
plt.show()

print("Saved figures:")
for path in sorted(Path("figures").glob("*.png")):
    print(" ", path)

## 19. Export and final audit

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value


# Export tables
headline_table.to_csv("results/headline_metrics.csv")
ablation_table.to_csv("results/ablation.csv")
scaling_table.to_csv("results/reservoir_scaling.csv", index=False)
encoding_table.to_csv("results/encoding_density.csv", index=False)
shot_table.to_csv("results/shot_budget.csv", index=False)
noise_table.to_csv("results/noise_zne.csv", index=False)
amplitude_damping_table.to_csv("results/amplitude_damping.csv", index=False)
forecast_predictions.to_csv("results/forecast_predictions.csv", index=False)
forecast_significance.to_csv("results/forecast_significance.csv", index=False)
transition_metrics.to_csv("results/transition_metrics.csv", index=False)
transition_confusion.to_csv("results/transition_confusion_matrix.csv")
if oxford_table is not None:
    oxford_table.to_csv("results/oxford_man_metrics.csv", index=False)

# Load hardware result if the hardware cell was run in an earlier session.
from pathlib import Path
import json

# Safely restore hardware variables after a kernel timeout or restart.
hardware_summary = globals().get("hardware_summary", None)
hardware_observables = globals().get("hardware_observables", None)

hardware_path = Path("results/hardware_validation.json")
observables_path = Path("results/hardware_observables.csv")

if hardware_summary is None and hardware_path.exists():
    hardware_summary = json.loads(
        hardware_path.read_text(encoding="utf-8")
    )
    print(
        "Loaded completed hardware summary from:",
        hardware_path
    )

elif hardware_summary is None:
    print(
        "No complete hardware_validation.json was found. "
        "The final audit will treat full hardware validation as optional."
    )

# Identify separately recovered IBM jobs.
recovered_metadata_files = sorted(
    Path("results").glob(
        "recovered_ibm_*_metadata.json"
    )
)

recovered_result_files = sorted(
    Path("results").glob(
        "recovered_ibm_*_evs.csv"
    )
)

recovered_hardware_jobs = []

for metadata_file in recovered_metadata_files:
    try:
        recovered_hardware_jobs.append(
            json.loads(
                metadata_file.read_text(
                    encoding="utf-8"
                )
            )
        )
    except Exception as exc:
        print(
            "Could not read",
            metadata_file,
            ":",
            exc,
        )

if recovered_hardware_jobs:
    print(
        f"Recovered IBM job records found: "
        f"{len(recovered_hardware_jobs)}"
    )

    for record in recovered_hardware_jobs:
        print(
            "-",
            record.get("job_id"),
            record.get("status"),
            record.get("backend"),
        )

if recovered_result_files:
    print("Recovered expectation-value files:")

    for result_file in recovered_result_files:
        print("-", result_file)

# Reference checks use tolerances because libraries and public-source rebuilds can create small drift.
reference = {
    "samples": 3750,
    "test_days": 750,
    "qrc_cal_rmse": 7.532e-5,
    "qrc_cal_qlike": 0.3261,
    "qrc_raw_rmse": 7.725e-5,
    "qrc_raw_qlike": 0.3794,
    "regime2_acc": 0.781,
    "majority_acc": 0.647,
    "oxford_qrc_rmse": 5.594e-5,
    "hardware_corr": 0.989,
    "hardware_mae": 0.0569,
}


def check_close(name, actual, expected, relative_tolerance=None, absolute_tolerance=None, note=""):
    if actual is None:
        return dict(Check=name, Status="SKIP", Actual="not available", Expected=expected, Note=note)
    if relative_tolerance is not None:
        passed = math.isclose(actual, expected, rel_tol=relative_tolerance, abs_tol=absolute_tolerance or 0.0)
    else:
        passed = math.isclose(actual, expected, abs_tol=absolute_tolerance or 0.0)
    return dict(Check=name, Status="PASS" if passed else "WARN", Actual=actual, Expected=expected, Note=note)


audit_rows = [
    check_close("Sample count", len(y), reference["samples"], absolute_tolerance=0),
    check_close("Test-day count", len(yt), reference["test_days"], absolute_tolerance=0),
    check_close("Calibrated QRC RMSE", results["QRC Dual+Pauli+FB (calibrated)"]["RMSE"], reference["qrc_cal_rmse"], relative_tolerance=0.02),
    check_close("Calibrated QRC QLIKE", results["QRC Dual+Pauli+FB (calibrated)"]["QLIKE"], reference["qrc_cal_qlike"], relative_tolerance=0.04),
    check_close("Raw QRC RMSE", results["QRC Dual+Pauli+FB"]["RMSE"], reference["qrc_raw_rmse"], relative_tolerance=0.02),
    check_close("Raw QRC QLIKE", results["QRC Dual+Pauli+FB"]["QLIKE"], reference["qrc_raw_qlike"], relative_tolerance=0.04),
    check_close("Calm/turbulent accuracy", regime2_acc, reference["regime2_acc"], absolute_tolerance=0.02),
    check_close("Majority baseline", majority_acc, reference["majority_acc"], absolute_tolerance=0.02),
]

if oxford_table is not None:
    oxford_qrc_rmse = float(oxford_table.loc[oxford_table.Model == "QRC Dual+Pauli+FB", "RMSE"].iloc[0])
    audit_rows.append(check_close("Oxford-Man QRC RMSE", oxford_qrc_rmse, reference["oxford_qrc_rmse"], relative_tolerance=0.03))
else:
    audit_rows.append(check_close("Oxford-Man QRC RMSE", None, reference["oxford_qrc_rmse"], note="Include data/oxman_spx.csv for the full submission."))

if hardware_summary:
    audit_rows.append(check_close("IBM feature correlation", hardware_summary.get("feature_correlation"), reference["hardware_corr"], absolute_tolerance=0.04, note=f"Compared with the locked Fez reference; current backend is {hardware_summary.get('backend', 'unknown')}. Actual replication values are retained."))
    audit_rows.append(check_close("IBM feature MAE", hardware_summary.get("feature_mae"), reference["hardware_mae"], absolute_tolerance=0.04, note=f"Compared with the locked Fez reference; current backend is {hardware_summary.get('backend', 'unknown')}. Actual replication values are retained."))
else:
    audit_rows.append(check_close("IBM feature correlation", None, reference["hardware_corr"], note="Optional for a judge rerun; include the team hardware JSON in the submission bundle."))

final_audit = pd.DataFrame(audit_rows)
final_audit.to_csv("results/final_audit.csv", index=False)

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "run_profile": RUN_PROFILE,
    "seed": SEED,
    "data_source": data_src,
    "market_data_sha256": MARKET_DATA_SHA256,
    "oxford_data_sha256": OXFORD_DATA_SHA256,
    "config": json_safe(CONFIG),
    "package_versions": VERSIONS,
    "reference_results": reference,
    "hardware_summary": hardware_summary,
    "hardware_replication_backend": CONFIG.get("ibm_backend"),
    "declared_primary_backend": CONFIG.get("declared_primary_backend"),
    "hardware_validation_role": CONFIG.get("hardware_validation_role"),
}
Path("results/run_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))
Path("results/requirements_runtime.txt").write_text(
    "\n".join(
        f"{package}=={version}"
        for package, version in VERSIONS.items()
        if version != "not installed"
    ) + "\n"
)

checklist = """QuantumEdge Phase 3 submission checklist
[ ] Notebook 1 executed with RUN_PROFILE='FULL' and RUN_HARDWARE_NOW=False
[ ] Notebook 4 official MNIST benchmark executed at 5/10/15 qubits
[ ] data/market_data.csv and data/oxman_spx.csv included
[ ] results/headline_metrics.csv and results/oxford_man_metrics.csv included
[ ] results/forecast_predictions.csv and results/forecast_significance.csv included
[ ] results/transition_metrics.csv included
[ ] results/noise_zne.csv and results/amplitude_damping.csv included
[ ] IBM hardware_validation.json and hardware_observables.csv included
[ ] README contains a working Launch on qBraid URL, not a placeholder
[ ] qBraid Skill included
[ ] official GIC cover page prepended without recreation
[ ] API keys/tokens absent
[ ] AI-use disclosure retained and team-reviewed
"""
Path("results/SUBMISSION_CHECKLIST.txt").write_text(checklist)

print("Final audit: PASS is expected for the committed-data FULL run. WARN indicates numerical drift that should be reviewed, not silently hidden.")
display(final_audit)
print("\nExported result files:")
for path in sorted(Path("results").glob("*")):
    print(" ", path)


## 20. Interpretation and disclosure

- The calibrated QRC is reported as the best RMSE/QLIKE variant on the locked Garman-Klass target.
- The uncalibrated QRC remains the architecture result used for the ablation and hardware-native claim.
- Mincer-Zarnowitz slopes above one are reported as residual bias rather than hidden.
- The Oxford-Man comparison is described as competitive with HAR-RV, not as a statistically established superiority claim.
- LSTM Sharpe may exceed the QRC Sharpe; the submission should not over-claim the trading metric.
- Flat size scaling and the failure of simulator-only entanglement features to help are reported as honest negative findings.

**AI-use disclosure:** Generative AI assisted with code review, notebook organization, and drafting. The architecture choices, experimental design, execution decisions, interpretation, and reported results remain the team's work. The team must review and re-voice submission prose in accordance with the competition rules.

# Appendix — Error handling for judges

| Symptom | Likely cause | Recovery |
|---|---|---|
| `FileNotFoundError` for a CSV | Folder structure changed | Place the notebook beside the submitted `data/` folder and rerun |
| Required column is missing | Wrong or modified CSV | Restore the submitted CSV from the original ZIP |
| Hash/cache warning | Data and feature cache do not match | Rebuild features or restore both submitted files together |
| `ModuleNotFoundError` | Environment not created | Install from `environment.yml` or `requirements.txt`, restart kernel |
| Kernel stops or memory error | Full feature generation exceeds session resources | Use `QUICK` or the validated cache |
| IBM authentication fails | No IBM entitlement or credentials | Skip hardware; inspect submitted hardware JSON and observables |
| `ibm_marrakesh` is unavailable | Maintenance, access restriction, entitlement, or queueing | Stop before submission and retain the ledger; do not silently substitute another device |
| Marrakesh output differs from Fez | Independent hardware, calibration, and noise conditions differ | Report the Marrakesh result separately and retain the actual values |
| Figure export fails | Output folder unavailable or previous cell failed | Create `figures/`, rerun the failed analysis cell, then rerun export |
| Final audit shows `WARN` | Numerical drift or incomplete optional section | Read the `Note` column and distinguish required from optional checks |

**Scientific fallback:** the executed notebook, exported CSV/JSON results, and graphical-results notebook permit inspection without consuming paid hardware time.

| `mnist_qrc_metrics.csv` missing | Mandatory MNIST notebook not yet run | Run Notebook 4 in `FULL` mode; do not substitute the sklearn digits smoke test |
| Launch button verifier failure | README still contains placeholder repository URL | Publish the repository and replace `REPLACE_WITH_PUBLIC_GITHUB_REPO.git` |
